ARTI308 - Machine Learning
# Lab 3: Exploratory Data Analysis (EDA)

EDA is the first and most important step in any Machine Learning project.
Before building models, we must understand:

- What does the data represent?
- Are there missing values?
- Are there outliers?
- What patterns exist?
- Which variables influence others?

If we do not understand the data, we cannot build a good model.

### Why EDA is Important

In real-world machine learning projects:

    1- 70–80% of the time is spent on understanding and cleaning data
    2- Only 20–30% is spent building models

**EDA helps us:**

- Detect data quality issues (missing values, duplicates, wrong types)
- Understand distributions and relationships between variables
- Identify outliers that could affect model performance
- Generate hypotheses before building machine learning models

In [ ]:
# Import Libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Make plots look cleaner
sns.set()
%matplotlib inline

## Loading data in different ways
### Option 1: Load local CSV

In [ ]:
# Load Dataset
df = pd.read_csv("Chocolate_Sales.csv")

# Display first 7 rows
df.head(7)

The dataset consists of individual chocolate sales records, where each row represents a single sales transaction.
The columns describe attributes such as the sales date, product information, sales location, and sales amount.
From the first few rows, we can observe that some columns contain textual data, while others contain numerical values stored as text, such as currency values (e.g. `$5,320.00`). This indicates that data type verification is necessary before further analysis.

### Option 2: Load online dataset using tensorflow.keras.datasets

In [ ]:
from tensorflow.keras import datasets
(train_images, train_labels), (test_images, test_labels) = datasets.cifar10.load_data()

print("Train images shape:", train_images.shape)
print("Train labels shape:", train_labels.shape)
print("Test images shape:",  test_images.shape)
print("Test labels shape:",  test_labels.shape)

This example demonstrates how datasets can be loaded directly from online sources using built-in libraries. The CIFAR-10 dataset is downloaded automatically and provided in predefined training and testing sets. This approach is commonly used in supervised learning experiments, especially for image-based datasets, as it simplifies data access and preparation.

### Option 3: Kaggle-style path reading

In [ ]:
# train_df = pd.read_csv("/kaggle/input/rsna-breast-cancer-detection/train.csv")
# test_df  = pd.read_csv("/kaggle/input/rsna-breast-cancer-detection/test.csv")

# train_df.head()

### Data type of columns

In [ ]:
# viewing the data types of columns
df.dtypes

The data type inspection shows that several columns are stored as `object` types, including `Date` and `Amount` (monetary values). While categorical features like `Country`, `Product`, and `Sales Person` are expected to be objects, the `Date` and `Amount` columns need to be converted to `datetime` and `numeric` types respectively before analysis.

In [ ]:
# Fix data types
df['Date']   = pd.to_datetime(df['Date'], dayfirst=True)
df['Amount'] = df['Amount'].replace('[\$,]', '', regex=True)
df['Amount'] = pd.to_numeric(df['Amount'])

df.dtypes

The `Date` column has been converted to a `datetime` format, and the `Amount` column has been cleaned (removing `$` and `,`) and converted to a numeric data type. This ensures that time-based analysis and arithmetic calculations can be performed correctly on these columns.

### Check Missing Values

In [ ]:
print(df.isna().sum())

The missing values analysis shows whether any features contain null or undefined values. In this dataset there are **no missing values** across all 6 columns, which means no imputation or row removal is required at this stage. This is a clean dataset ready for EDA.

### Check duplicate rows

In [ ]:
# checking duplicate rows
duplicates = df.duplicated()
print(f"Total duplicate rows: {duplicates.sum()}")
duplicates[duplicates == True]

The duplicate records check identifies whether the same sales transaction appears more than once. In this dataset there are **0 duplicate rows**, meaning every transaction is unique. No rows need to be dropped due to duplication.

### No. of rows and columns

In [ ]:
# finding number of rows and columns
print("Shape (rows, columns): ", df.shape, "\n")
print("Number of rows:    ", df.shape[0])
print("Number of columns: ", df.shape[1])

The dataset consists of **3,282 rows** and **6 columns**. This size is suitable for exploratory analysis and allows meaningful insights without being computationally expensive. Knowing the dataset dimensions helps in selecting appropriate analysis and visualization techniques.

### Descriptive Summary Statistics

In [ ]:
# Statistical summary
df.describe(include='all')

The descriptive statistics provide an overview of both numerical and categorical features:
- **Amount** ranges from \$7 to \$26,170 with a mean of ~\$6,030, indicating high variability in transaction values.
- **Boxes Shipped** ranges from 1 to 778 with a mean of ~165 boxes per transaction.
- The dataset covers sales across **6 countries**, **25 salespersons**, and multiple products.
- The date range spans from **January 2022 to August 2024**.

### Univariate Analysis

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(df['Boxes Shipped'], bins=20, color='steelblue', edgecolor='white')
plt.title("Distribution of Boxes Shipped", fontsize=14)
plt.xlabel("Boxes Shipped")
plt.ylabel("Frequency")
plt.show()

- Shows how shipment sizes are distributed
- **Right-skewed** distribution: most transactions involve a relatively small number of boxes (< 200), while a few involve very large shipments (> 500).

This skewness is common in sales data and may influence later modeling decisions, such as applying log transformation before training a regression model.

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(df['Amount'], bins=20, color='darkorange', edgecolor='white')
plt.title("Distribution of Revenue", fontsize=14)
plt.xlabel("Revenue ($)")
plt.ylabel("Frequency")
plt.show()

The revenue distribution is also **right-skewed**, with most transactions generating moderate revenue (under \$10,000) and fewer transactions generating very high revenue (above \$20,000). This suggests the presence of high-value sales that may act as outliers or important contributors to total revenue.

## Bivariate Analysis

### Revenue by Country

In [ ]:
country_revenue = df.groupby('Country')['Amount'].sum().sort_values(ascending=False)

plt.figure(figsize=(10,5))
country_revenue.plot(kind='bar', color='teal', edgecolor='black')
plt.title("Total Revenue by Country", fontsize=14)
plt.ylabel("Total Revenue ($)")
plt.xlabel("Country")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print(country_revenue)

The total revenue varies across countries. Some countries contribute significantly more to overall sales than others. This indicates **geographical differences in sales performance** and suggests that `Country` is an important categorical feature that should be retained for modeling.

In [ ]:
product_revenue = df.groupby('Product')['Amount'].sum().sort_values(ascending=False)

plt.figure(figsize=(12,5))
product_revenue.plot(kind='bar', color='coral', edgecolor='black')
plt.title("Revenue by Product", fontsize=14)
plt.ylabel("Total Revenue ($)")
plt.xlabel("Product")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print(product_revenue)

The revenue distribution across products shows that certain chocolate products generate higher total revenue than others. This highlights **product-level performance differences** and suggests that `Product` is a significant feature in the dataset.

In [ ]:
salesperson_revenue = df.groupby('Sales Person')['Amount'].sum().sort_values(ascending=False)

plt.figure(figsize=(12,5))
salesperson_revenue.plot(kind='bar', color='mediumpurple', edgecolor='black')
plt.title("Revenue by Sales Person", fontsize=14)
plt.ylabel("Total Revenue ($)")
plt.xlabel("Sales Person")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

salesperson_revenue.head(10)

The revenue contribution varies among salespersons, with a few individuals generating notably higher total revenue. This may reflect differences in sales regions, experience, or customer base. The `Sales Person` feature could serve as a useful grouping variable in analysis.

### Boxes vs Revenue Relationship

In [ ]:
plt.figure(figsize=(8,5))
sns.scatterplot(x='Boxes Shipped', y='Amount', data=df, alpha=0.5, color='steelblue')
plt.title("Boxes Shipped vs Revenue", fontsize=14)
plt.xlabel("Boxes Shipped")
plt.ylabel("Revenue ($)")
plt.show()

The scatter plot shows a **general positive relationship** between the number of boxes shipped and revenue. However, the spread of points indicates variability — revenue is influenced by additional factors beyond shipment volume, such as product type or pricing.

### Correlation Matrix

In [ ]:
plt.figure(figsize=(6,4))
sns.heatmap(df[['Boxes Shipped', 'Amount']].corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title("Correlation Matrix", fontsize=13)
plt.show()

- Correlation close to **1** = strong positive relationship
- Close to **0** = weak or no relationship

The correlation matrix shows a **moderate positive correlation** between boxes shipped and revenue. While shipment volume contributes to revenue, it is not the sole determining factor — other features like product type and country likely explain additional variance.

## Time-Based Analysis
### Monthly Revenue Trend

In [ ]:
df['Month'] = df['Date'].dt.to_period('M')

monthly_revenue = df.groupby('Month')['Amount'].sum()

plt.figure(figsize=(12,5))
monthly_revenue.plot(color='darkgreen', linewidth=2)
plt.title("Monthly Revenue Trend", fontsize=14)
plt.ylabel("Total Revenue ($)")
plt.xlabel("Month")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

The monthly revenue trend shows **fluctuations over time**, indicating seasonal or periodic variations in sales. Some months exhibit spikes while others show drops. Identifying such trends is useful for understanding business cycles and can inform forecasting and time-series modeling in future analysis.

## Additional Analysis
### Revenue Distribution by Country (Boxplot)

In [ ]:
plt.figure(figsize=(10,5))
sns.boxplot(x='Country', y='Amount', data=df, palette='Set2')
plt.title("Revenue Distribution by Country", fontsize=14)
plt.xlabel("Country")
plt.ylabel("Revenue ($)")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

The boxplot reveals how revenue is distributed within each country. The median, interquartile range (IQR), and outliers (dots above/below whiskers) are clearly visible. Some countries show wider spread and more outliers, indicating more variability in their transaction values.

In [ ]:
plt.figure(figsize=(10,5))
sns.boxplot(x='Country', y='Boxes Shipped', data=df, palette='pastel')
plt.title("Boxes Shipped Distribution by Country", fontsize=14)
plt.xlabel("Country")
plt.ylabel("Boxes Shipped")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## EDA Summary

| Finding | Detail |
|---|---|
| **Dataset size** | 3,282 rows × 6 columns |
| **Missing values** | None |
| **Duplicate rows** | None |
| **Date range** | Jan 2022 – Aug 2024 |
| **Revenue range** | \$7 – \$26,170 (mean: ~\$6,030) |
| **Boxes shipped range** | 1 – 778 (mean: ~165) |
| **Countries** | 6 (UK, India, Australia, NZ, USA, Canada) |
| **Unique products** | 22 |
| **Salespersons** | 25 |
| **Key insight** | Both Amount and Boxes Shipped are right-skewed |
| **Correlation** | Moderate positive correlation between Boxes Shipped and Revenue |

**Next Steps (after EDA):**
- Encode categorical features (Country, Product, Sales Person)
- Apply log transformation to skewed numerical features
- Split data into train/test sets
- Build and evaluate a regression or classification model

# Assignment

In this assignment, you will apply the EDA techniques learned in class to a dataset of your choice. You must submit the dataset file with your notebook.

End of lab 3.